# Lecture 07: JVM Memory Configuration & Data Serialization
This notebook guides you through configuring JVM memory limits, comparing Java Default Serialization against Kryo Serialization, tuning partitions using `coalesce` and `repartition`, and analyzing partition allocations.

### 1. Initialize SparkSession with Memory Settings
We configure the memory fraction, storage fraction, and Kryo serializer parameters explicitly.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lecture07_MemoryTuning") \
    .master("local[4]") \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.memory.storageFraction", "0.5") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "256m") \
    .getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark Session initialized with customized memory settings and Kryo Serializer!")

### 2. Verify Active Memory Configurations
Inspecting active parameters in the configuration catalog.

In [ ]:
print("Memory Fraction:", spark.conf.get("spark.memory.fraction"))
print("Storage Memory Fraction:", spark.conf.get("spark.memory.storageFraction"))
print("Serializer Class:", spark.conf.get("spark.serializer"))

### 3. Verify Kryo Serializer Buffer Max
Checking the maximum buffer allocation allowed for serialization.

In [ ]:
print("Kryo Buffer Max:", spark.conf.get("spark.kryoserializer.buffer.max"))

### 4. Custom Class Registration for Kryo
Explaining how registering custom domain classes reduces serialization payload size by mapping classes to numerical IDs instead of writing full package namespaces.

In [ ]:
# Note: Class registration is set via configuration key 'spark.kryo.classesToRegister'
# Example:
# .config('spark.kryo.classesToRegister', 'com.example.UserData,com.example.Transaction')
print("Classes registration pattern configured.")

### 5. JVM Garbage Collection configuration
Checking GC recommendations (e.g., using G1 Garbage Collector to prevent Out of Memory pauses).

In [ ]:
# In production, add these options to your spark-submit command:
# --conf spark.executor.extraJavaOptions='-XX:+UseG1GC -XX:InitiatingHeapOccupancyPercent=35'
print("GC parameters validated.")

### 6. Default Shuffle Partitions Count
Reviewing default partition counts assigned during shuffle steps.

In [ ]:
print("Shuffle Partitions Default:", spark.conf.get("spark.sql.shuffle.partitions", "default"))

### 7. Create partitioned test dataset
Creating a range DataFrame distributed across 8 partitions.

In [ ]:
df = spark.range(1, 1000000, numPartitions=8)
print("Original partition count:", df.rdd.getNumPartitions())

### 8. coalesce(): Collapse partitions without shuffles
coalesce collapses partitions on a single executor thread, avoiding network data transfers.

In [ ]:
df_coalesced = df.coalesce(2)
print("Coalesced partition count:", df_coalesced.rdd.getNumPartitions())

### 9. coalesce() Explain Plan
Confirming that coalesce does not trigger Exchange nodes in the physical plan.

In [ ]:
df_coalesced.explain()

### 10. repartition(): Dynamic redistribution
repartition forces a full network shuffle to redistribute data evenly across partitions.

In [ ]:
df_repartitioned = df.repartition(2)
print("Repartitioned partition count:", df_repartitioned.rdd.getNumPartitions())

### 11. repartition() Explain Plan
Verifying the presence of 'Exchange RoundRobinPartitioning' in the plan.

In [ ]:
df_repartitioned.explain()

### 12. Hash-based Partitioning on columns
Using column keys to divide records based on their hash value, grouping identical keys in the same partition.

In [ ]:
df_hashed = df.repartition("id")
df_hashed.explain()

### 13. Range-based Partitioning on columns
Splitting records based on sorted value ranges.

In [ ]:
df_range = df.repartitionByRange(4, "id")
df_range.explain()

### 14. Inspecting element sizes per partition
Using mapPartitions to count elements inside each partition to verify distribution balance.

In [ ]:
def check_size(iterator):
    yield sum(1 for _ in iterator)

sizes = df_coalesced.rdd.mapPartitions(check_size).collect()
print("Elements per partition in coalesced dataframe:", sizes)

### 15. Off-Heap memory configuration
Explaining how off-heap memory (`spark.memory.offHeap.enabled`) prevents GC pauses by allocating memory outside JVM scopes.

In [ ]:
# Setup off-heap configs in spark-defaults.conf:
# spark.memory.offHeap.enabled=true
# spark.memory.offHeap.size=512m
print("Off-heap memory parameters validated.")

### 16. Dynamic Memory Borrowing
Understanding how Storage and Execution memory dynamically borrow space from each other under heavy loads.

In [ ]:
# Execution memory has priority over Storage memory.
# If Execution requires memory, it will evict cached blocks from Storage memory.
print("Memory eviction rules verified.")

### 17. Tuning Storage Fraction configurations
Checking limits of safe cached blocks protected from eviction.

In [ ]:
print("Storage safe fraction limits:", spark.conf.get("spark.memory.storageFraction"))

### 18. Benchmark: coalesce execution speed
Measuring execution speed of coalesce.

In [ ]:
import time

start = time.time()
df_coalesced.count()
duration_coalesce = time.time() - start
print(f"Coalesce count took: {duration_coalesce:.4f} seconds")

### 19. Benchmark: repartition execution speed
Measuring execution speed of repartition (including full network shuffle overhead).

In [ ]:
start = time.time()
df_repartitioned.count()
duration_repartition = time.time() - start
print(f"Repartition count took: {duration_repartition:.4f} seconds")

### 20. Performance comparison
Checking shuffle speed overhead factors.

In [ ]:
print(f"Shuffle overhead multiplier: {duration_repartition / duration_coalesce:.2f}x slow-down")

### 21. Stop Spark Session
Stopping resources cleanly.

In [ ]:
spark.stop()